In [1]:
import groq
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")

print("Groq key loaded:", "✅" if groq_key else "❌ MISSING")
print("Google key loaded:", "✅" if google_key else "❌ MISSING")

Groq key loaded: ✅
Google key loaded: ✅


c:\Users\user\OneDrive\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_24396\3241140904.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()

# Configure Gemini
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Test with a fake transcript
question = "Explain what a fraction is"
transcript = "A fraction is a part of a whole number. Like one half means you split something into two parts and take one."

prompt = f"""You are evaluating a middle school student's math explanation.

The teacher asked: {question}

The student explained verbally: {transcript}

Respond in this EXACT format:
SCORE: [1, 2, 3, or 4]
WHAT_WAS_RIGHT: [one encouraging sentence]
WHAT_TO_IMPROVE: [one specific suggestion]
TEACHER_NOTE: [one sentence for the teacher]
LANGUAGE_DETECTED: [English / Roman Urdu / Sindhi / Mixed]"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

SCORE: 3
WHAT_WAS_RIGHT: Your example of "one half" clearly shows you understand how fractions represent taking parts of a whole!
WHAT_TO_IMPROVE: To be even more precise, explain that a fraction is a part of a *whole* (like a pizza or a group of items), rather than just a "whole number."
TEACHER_NOTE: The student's strong example indicates a good conceptual grasp, which can be built upon to refine their formal definition.
LANGUAGE_DETECTED: English


In [5]:
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from datetime import datetime

# Create database engine - this creates voicecheck.db file
engine = create_engine("sqlite:///voicecheck.db", echo=True)

Base = declarative_base()

# Sessions table - one row per teacher question
class Session(Base):
    __tablename__ = "sessions"
    
    id = Column(Integer, primary_key=True)
    question = Column(Text, nullable=False)
    student_link = Column(String, unique=True)
    created_at = Column(DateTime, default=datetime.utcnow)

# Responses table - one row per student submission
class Response(Base):
    __tablename__ = "responses"
    
    id = Column(Integer, primary_key=True)
    session_id = Column(Integer, nullable=False)
    audio_filename = Column(String)
    canvas_image_filename = Column(String)
    uploaded_image_filename = Column(String, nullable=True)
    transcript = Column(Text)
    score = Column(Integer)
    what_was_right = Column(Text)
    what_to_improve = Column(Text)
    teacher_note = Column(Text)
    language_detected = Column(String)
    submitted_at = Column(DateTime, default=datetime.utcnow)

# Create all tables
Base.metadata.create_all(engine)
print("✅ Database and tables created successfully!")

2026-04-29 23:02:56,007 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-29 23:02:56,008 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("sessions")
2026-04-29 23:02:56,009 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-29 23:02:56,011 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("sessions")
2026-04-29 23:02:56,012 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-29 23:02:56,012 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("responses")
2026-04-29 23:02:56,013 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-29 23:02:56,015 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("responses")
2026-04-29 23:02:56,016 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-29 23:02:56,018 INFO sqlalchemy.engine.Engine 
CREATE TABLE sessions (
	id INTEGER NOT NULL, 
	question TEXT NOT NULL, 
	student_link VARCHAR, 
	created_at DATETIME, 
	PRIMARY KEY (id), 
	UNIQUE (student_link)
)


2026-04-29 23:02:56,019 INFO sqlalchemy.engine.Engine [no key 0.00089s] ()
2026-0

C:\Users\user\AppData\Local\Temp\ipykernel_24396\811364874.py:9: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [7]:
from sqlalchemy.orm import sessionmaker
import uuid

# Create a session to talk to the database
SessionLocal = sessionmaker(bind=engine)
db = SessionLocal()

# Insert a fake teacher session
fake_session = Session(
    question="Explain what a fraction is",
    student_link=str(uuid.uuid4()),
    created_at=datetime.utcnow()
)

db.add(fake_session)
db.commit()
db.refresh(fake_session)

print("✅ Session inserted!")
print("Session ID:", fake_session.id)
print("Question:", fake_session.question)
print("Student Link:", fake_session.student_link)
print("Created At:", fake_session.created_at)

2026-04-29 23:04:45,817 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-29 23:04:45,819 INFO sqlalchemy.engine.Engine INSERT INTO sessions (question, student_link, created_at) VALUES (?, ?, ?)
2026-04-29 23:04:45,821 INFO sqlalchemy.engine.Engine [cached since 4.663s ago] ('Explain what a fraction is', '0ba22348-1e5c-4a43-a637-3ef7ec841206', '2026-04-29 18:04:45.812980')
2026-04-29 23:04:45,828 INFO sqlalchemy.engine.Engine COMMIT
2026-04-29 23:04:45,836 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-29 23:04:45,838 INFO sqlalchemy.engine.Engine SELECT sessions.id, sessions.question, sessions.student_link, sessions.created_at 
FROM sessions 
WHERE sessions.id = ?
2026-04-29 23:04:45,840 INFO sqlalchemy.engine.Engine [cached since 4.654s ago] (2,)
✅ Session inserted!
Session ID: 2
Question: Explain what a fraction is
Student Link: 0ba22348-1e5c-4a43-a637-3ef7ec841206
Created At: 2026-04-29 18:04:45.812980


C:\Users\user\AppData\Local\Temp\ipykernel_24396\2106171592.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow()


In [10]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version()"))
    print("✅ Connected to Supabase!")
    print(result.fetchone()[0])

OperationalError: (psycopg2.OperationalError) could not translate host name "db.dmqruveoffolfkhsyqjg.supabase.co" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)